# LLM Prompting Methods for Healthcare Text & Structured Data

This tutorial compares four prompting strategies — **Zero-shot, Few-shot, Chain-of-Thought (CoT), and Tree-of-Thoughts (ToT)** — on the *same* classification goal across two different data shapes:

1. **MTSamples** (free-text clinical transcriptions) → predict `medical_specialty`
2. **Synthea** (structured synthetic EHR: conditions + medications) → predict opioid-misuse risk (Low/High), using a later recorded "Drug overdose" condition as a proxy ground-truth label

**Ground truth**
- MTSamples: the real `medical_specialty` column already in the dataset (not invented).
- Synthea: a *proxy* label — patients with a later recorded "Drug overdose" condition are labeled High risk. This is bounded by what the Synthea simulator generated, not a validated clinical risk score. Stated explicitly here for transparency.

**Data note:** MTSamples and Synthea CSVs are both public/synthetic and contain no real patient data, consistent with assignment requirements.


## 1. Setup

In [ ]:
!pip install openai pandas scikit-learn --quiet


In [ ]:
import os
import json
import time
import pandas as pd
from openai import OpenAI

# Set your API key as an environment variable before running:
#   export OPENAI_API_KEY="sk-..."
# or set it directly here (not recommended for shared notebooks):
# os.environ["OPENAI_API_KEY"] = "sk-..."

client = OpenAI()  # reads OPENAI_API_KEY from environment
MODEL = "gpt-4o-mini"


## 2. Load MTSamples eval set + ground truth

`mtsamples_eval_set.csv` is a stratified sample: 7 records each from 7 specialties
(Surgery, Cardiovascular/Pulmonary, Orthopedic, Radiology, Neurology, Urology,
Gastroenterology) — 49 records total.

`mtsamples_fewshot_pool.csv` holds a separate, non-overlapping set of 14 records
(2 per specialty) used only to build few-shot examples.


In [ ]:
mt_eval = pd.read_csv("mtsamples_eval_set.csv")
mt_fewshot_pool = pd.read_csv("mtsamples_fewshot_pool.csv")

SPECIALTIES = sorted(mt_eval["medical_specialty"].unique().tolist())

print("Ground truth — MTSamples specialty counts (eval set):")
print(mt_eval["medical_specialty"].value_counts())
print("\nTotal eval records:", len(mt_eval))
print("Specialty label set:", SPECIALTIES)


## 3. Load Synthea eval set + ground truth

`synthea_eval_ids.json` holds patient IDs for 25 overdose-positive and 25
negative patients (eval set), plus a separate 4+4 few-shot pool, and each
positive patient's overdose date (used to truncate their history so the
model never sees the overdose itself — avoiding label leakage).


In [ ]:
patients = pd.read_csv("synthea_data/csv/patients.csv")
conditions = pd.read_csv("synthea_data/csv/conditions.csv")
medications = pd.read_csv("synthea_data/csv/medications.csv")

with open("synthea_eval_ids.json") as f:
    synthea_ids = json.load(f)

overdose_dates = synthea_ids["overdose_dates"]

def build_patient_text(patient_id, truncate_before=None):
    """Build a plain-text timeline of a patient's conditions + medications.
    If truncate_before is set (a date string), only include events strictly
    before that date -- used to avoid leaking the overdose outcome itself."""
    c = conditions[conditions["PATIENT"] == patient_id][["START", "DESCRIPTION"]].copy()
    m = medications[medications["PATIENT"] == patient_id][["START", "DESCRIPTION"]].copy()
    c["START"] = c["START"].astype(str).str[:10]
    m["START"] = m["START"].astype(str).str[:10]

    if truncate_before:
        cutoff = truncate_before[:10]
        c = c[c["START"] < cutoff]
        m = m[m["START"] < cutoff]

    c = c.sort_values("START").drop_duplicates("DESCRIPTION")
    m = m.sort_values("START").drop_duplicates("DESCRIPTION")

    lines = ["CONDITIONS (chronological):"]
    for _, r in c.iterrows():
        lines.append(f"  {r['START']}: {r['DESCRIPTION']}")
    lines.append("MEDICATIONS (chronological):")
    for _, r in m.iterrows():
        lines.append(f"  {r['START']}: {r['DESCRIPTION']}")
    return "\n".join(lines)

# Build eval records: (patient_id, text, ground_truth_label)
synthea_eval = []
for pid in synthea_ids["eval_positive"]:
    txt = build_patient_text(pid, truncate_before=overdose_dates.get(pid))
    synthea_eval.append({"patient_id": pid, "text": txt, "ground_truth": "High"})
for pid in synthea_ids["eval_negative"]:
    txt = build_patient_text(pid, truncate_before=None)
    synthea_eval.append({"patient_id": pid, "text": txt, "ground_truth": "Low"})

synthea_eval_df = pd.DataFrame(synthea_eval)

print("Ground truth — Synthea risk label counts (eval set):")
print(synthea_eval_df["ground_truth"].value_counts())
print("\nTotal eval patients:", len(synthea_eval_df))


## 4. Build few-shot example blocks

Few-shot examples are drawn only from the separate pools (never the eval set)
to avoid the model simply memorizing eval answers.


In [ ]:
# --- MTSamples few-shot examples ---
def build_mtsamples_fewshot_block(n_per_class=1):
    lines = []
    for spec in SPECIALTIES:
        rows = mt_fewshot_pool[mt_fewshot_pool["medical_specialty"] == spec].head(n_per_class)
        for _, r in rows.iterrows():
            snippet = str(r["transcription"])[:600]
            lines.append(f'Report: """{snippet}"""\nSpecialty: {spec}\n')
    return "\n".join(lines)

mt_fewshot_block = build_mtsamples_fewshot_block(n_per_class=1)

# --- Synthea few-shot examples ---
def build_synthea_fewshot_block():
    lines = []
    for pid in synthea_ids["fewshot_positive"]:
        txt = build_patient_text(pid, truncate_before=overdose_dates.get(pid))
        lines.append(f'Patient history:\n{txt}\nRisk: High\n')
    for pid in synthea_ids["fewshot_negative"]:
        txt = build_patient_text(pid, truncate_before=None)
        lines.append(f'Patient history:\n{txt}\nRisk: Low\n')
    return "\n".join(lines)

synthea_fewshot_block = build_synthea_fewshot_block()
print(mt_fewshot_block[:800])


## 5. Prompt templates

Every method / dataset combination returns the **same fixed JSON schema**:
`{"label": "...", "reason": "..."}` — this is what makes the four methods
directly comparable.


In [ ]:
def mtsamples_prompt(method, transcription):
    schema_note = 'Respond ONLY with JSON: {"label": "<one specialty>", "reason": "<one sentence>"}'
    specialty_list = ", ".join(SPECIALTIES)

    if method == "zero_shot":
        return f"""Classify the medical specialty of this clinical report.
Choose exactly one from: {specialty_list}.
{schema_note}

Report:
\"\"\"{transcription}\"\"\""""

    elif method == "few_shot":
        return f"""Classify the medical specialty of a clinical report.
Choose exactly one from: {specialty_list}.
{schema_note}

Examples:
{mt_fewshot_block}

Now classify this report:
Report: \"\"\"{transcription}\"\"\""""

    elif method == "cot":
        return f"""Classify the medical specialty of this clinical report.
Choose exactly one from: {specialty_list}.

First, silently reason step by step: (1) identify the key clinical findings,
(2) identify the procedure or diagnosis type, (3) match those to the closest
specialty. Then output ONLY the final JSON (do not show your reasoning steps
in the output).
{schema_note}

Report:
\"\"\"{transcription}\"\"\""""

    elif method == "tot":
        return f"""Classify the medical specialty of this clinical report.
Choose exactly one from: {specialty_list}.

Internally consider THREE plausible specialty interpretations based on the
report's findings, briefly weigh the evidence for each, then select the
best-supported one. Output ONLY the final JSON (do not show the three
interpretations in the output).
{schema_note}

Report:
\"\"\"{transcription}\"\"\""""

    else:
        raise ValueError(method)


def synthea_prompt(method, patient_text):
    schema_note = ('Respond ONLY with JSON: {"label": "Low" or "High", '
                    '"probability_high": <float 0.0-1.0, your confidence that risk is High>, '
                    '"reason": "<one sentence>"}')

    if method == "zero_shot":
        return f"""Given this patient's condition and medication history, classify
their opioid-misuse risk as Low or High.
{schema_note}

{patient_text}"""

    elif method == "few_shot":
        return f"""Given a patient's condition and medication history, classify
their opioid-misuse risk as Low or High.
{schema_note}

Examples:
{synthea_fewshot_block}

Now classify this patient:
{patient_text}"""

    elif method == "cot":
        return f"""Given this patient's condition and medication history, classify
their opioid-misuse risk as Low or High.

First, silently reason step by step: (1) list the conditions in chronological
order, (2) list the medications in chronological order, (3) note any pattern
of escalation, switching, or discontinuation in medication type/potency over
time, (4) classify risk based on that pattern. Then output ONLY the final
JSON (do not show your reasoning steps in the output).
{schema_note}

{patient_text}"""

    elif method == "tot":
        return f"""Given this patient's condition and medication history, classify
their opioid-misuse risk as Low or High.

Internally consider THREE possible explanations for this timeline:
(a) normal short-term pain management, (b) tolerance-driven escalation,
(c) unrelated/incidental events. Briefly weigh each against the timeline,
then classify risk based on the best-supported explanation. Output ONLY the
final JSON (do not show the three explanations in the output).
{schema_note}

{patient_text}"""

    else:
        raise ValueError(method)


## 6. API call + parsing helper

Uses JSON mode where available and retries once on malformed output.


In [ ]:
def call_llm(prompt, retries=2):
    """Calls the model and returns the full parsed JSON dict (label, reason,
    and -- for Synthea prompts -- probability_high). Returns a dict with an
    'error' key on failure instead of raising, so a batch run doesn't stop."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                response_format={"type": "json_object"},
            )
            content = resp.choices[0].message.content
            parsed = json.loads(content)
            parsed["label"] = str(parsed.get("label", "")).strip()
            parsed["reason"] = str(parsed.get("reason", "")).strip()
            if "probability_high" in parsed:
                try:
                    parsed["probability_high"] = float(parsed["probability_high"])
                except (TypeError, ValueError):
                    parsed["probability_high"] = None
            return parsed
        except Exception as e:
            if attempt == retries:
                print("Failed after retries:", e)
                return {"label": "PARSE_ERROR", "reason": str(e), "probability_high": None}
            time.sleep(1)


## 7. Run all methods on MTSamples (49 records x 4 methods = 196 calls)

This will take a few minutes and cost well under $1 on gpt-4o-mini.


In [ ]:
METHODS = ["zero_shot", "few_shot", "cot", "tot"]

mt_results = []
for _, row in mt_eval.iterrows():
    for method in METHODS:
        prompt = mtsamples_prompt(method, row["transcription"])
        parsed = call_llm(prompt)
        mt_results.append({
            "record_id": row["sample_name"],
            "method": method,
            "predicted_label": parsed["label"],
            "ground_truth": row["medical_specialty"],
            "reason": parsed["reason"],
        })

mt_results_df = pd.DataFrame(mt_results)
mt_results_df.to_csv("mtsamples_results.csv", index=False)
mt_results_df.head()


## 8. Run all methods on Synthea (50 patients x 4 methods = 200 calls)


In [ ]:
synthea_results = []
for _, row in synthea_eval_df.iterrows():
    for method in METHODS:
        prompt = synthea_prompt(method, row["text"])
        parsed = call_llm(prompt)
        synthea_results.append({
            "patient_id": row["patient_id"],
            "method": method,
            "predicted_label": parsed["label"],
            "probability_high": parsed.get("probability_high"),
            "ground_truth": row["ground_truth"],
            "reason": parsed["reason"],
        })

synthea_results_df = pd.DataFrame(synthea_results)
synthea_results_df.to_csv("synthea_results.csv", index=False)
synthea_results_df.head()


## 9. Summary Table 1 — MTSamples: predicted specialty counts by method


In [ ]:
mt_summary_counts = pd.crosstab(mt_results_df["method"], mt_results_df["predicted_label"])
print("Predicted label distribution by method:")
mt_summary_counts


In [ ]:
mt_accuracy = (
    mt_results_df.groupby("method")
    .apply(lambda g: (g["predicted_label"] == g["ground_truth"]).mean())
    .rename("accuracy")
    .reset_index()
)
print("Accuracy by prompting method (MTSamples specialty classification):")
mt_accuracy


## 10. Summary Table 2 — Synthea: predicted risk counts + confusion matrix by method


In [ ]:
synthea_summary_counts = pd.crosstab(synthea_results_df["method"], synthea_results_df["predicted_label"])
print("Predicted label distribution by method:")
synthea_summary_counts


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score

rows = []
for method in METHODS:
    sub = synthea_results_df[synthea_results_df["method"] == method]
    y_true = (sub["ground_truth"] == "High").astype(int)
    y_pred = (sub["predicted_label"] == "High").astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    rows.append({
        "method": method,
        "TP (correctly caught High risk)": tp,
        "FN (missed High risk)": fn,
        "FP (false alarm)": fp,
        "TN (correctly Low risk)": tn,
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 2),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 2),
        "accuracy": round(accuracy_score(y_true, y_pred), 2),
    })

synthea_confusion_summary = pd.DataFrame(rows)
print("Confusion matrix + precision/recall/accuracy by method (Synthea opioid-risk proxy):")
synthea_confusion_summary


## 10b. AUROC / AUPRC by method

Same style of evaluation used in the course's drug-synergy example (Section 4:
embeddings + logistic regression, scored with AUROC/AUPRC). Here we use the
model's own `probability_high` confidence score instead of a classifier's
predicted probability, but the metric and interpretation are the same:
AUROC/AUPRC reward a method for correctly *ranking* patients by risk, not
just picking the right label at the 0.5 threshold -- a stricter test than
accuracy alone, and especially informative given the class imbalance in the
underlying Synthea data (52/1,171 true positives).


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

auc_rows = []
for method in METHODS:
    sub = synthea_results_df[synthea_results_df["method"] == method].dropna(subset=["probability_high"])
    if sub["probability_high"].nunique() < 2 or sub["ground_truth"].nunique() < 2:
        auc_rows.append({"method": method, "AUROC": None, "AUPRC": None,
                          "note": "insufficient variation to compute"})
        continue
    y_true = (sub["ground_truth"] == "High").astype(int)
    y_score = sub["probability_high"]
    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    auc_rows.append({"method": method, "AUROC": round(auroc, 3), "AUPRC": round(auprc, 3), "note": ""})

synthea_auc_summary = pd.DataFrame(auc_rows)
print("AUROC / AUPRC by prompting method (Synthea opioid-risk proxy):")
synthea_auc_summary


## 11. Caveats (include on your evaluation slide)

- **MTSamples ground truth** is the dataset's own `medical_specialty` field —
  a real label, not invented for this project.
- **Synthea ground truth is a proxy**, not a validated clinical risk score:
  "High risk" = the simulated patient happened to have a later recorded
  "Drug overdose" condition in this specific Synthea run. Absence of that
  condition does not prove the patient was truly low-risk — Synthea's
  simulation may simply not have generated that outcome for them.
- **Positive cases were oversampled for evaluation.** Only 52 of 1,171
  patients (4.4%) in the underlying Synthea sample have a recorded overdose;
  the eval set uses 25 positives + 25 negatives so every method has a fair
  chance to be tested on both classes, rather than the natural imbalance
  hiding nearly all positives.
- All few-shot examples were drawn from a separate pool with zero overlap
  with the evaluation records, to avoid memorization inflating results.
